# Model Training

## Auto ML

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import h2o
from h2o.automl import H2OAutoML

In [19]:
df_pandas = pd.read_excel("trainable_dataset.xlsx", engine='openpyxl', dtype={'Risultato': str})
df_pandas['Risultato'] = df_pandas['Risultato'].replace('X', '3')

In [20]:
print(df_pandas.head())

  Risultato  Ball possession_Casa  Expected goals (xG)_Casa  \
0         2                    47                      0.70   
1         2                    48                      0.89   
2         2                    32                      0.21   
3         1                    48                      2.69   
4         3                    60                      1.90   

   Expected goals (xG)_Fuori  Accurate passes_Casa  Accurate passes_Fuori  \
0                       0.51                    64                     71   
1                       1.50                    88                     89   
2                       0.99                    75                     88   
3                       0.78                    90                     90   
4                       0.14                    82                     73   

   Shots inside box_Casa  Shots inside box_Fuori  Shots outside box_Casa  \
0                      4                       6                       6   
1     

In [21]:
h2o.init()
# togliamo eventuali spazi vuoti nei nomi
df_pandas.columns = df_pandas.columns.str.strip()
df = h2o.H2OFrame(df_pandas)

target = 'Risultato'

# as factor categorizza la variabile 
df[target] = df[target].asfactor()

ignore_columns = ['Home_ID', 'Away_ID', target]
predictors = [col for col in df.columns if col not in ignore_columns]

# Suddivisione dei dati in Train (80%) e Test (20%)
train, test = df.split_frame(ratios=[0.8], seed=42)

# inizializza e avvia H2O AutoML
print("Inizio l'addestramento con H2O AutoML...")
# max_models=20 dice ad H2O di addestrare un massimo di 20 modelli base 
aml = H2OAutoML(max_models=20, seed=42, sort_metric="mean_per_class_error") 

# Avvia l'addestramento
aml.train(x=predictors, y=target, training_frame=train)

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,36 mins 42 secs
H2O_cluster_timezone:,Europe/Rome
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,27 days
H2O_cluster_name:,H2O_from_python_Giacomo_f60jb3
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.846 Gb
H2O_cluster_total_cores:,12
H2O_cluster_allowed_cores:,12
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Inizio l'addestramento con H2O AutoML...
AutoML progress: |█
16:50:27.260: AutoML: XGBoost is not available; skipping it.

██████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),5/5
# GBM base models (used / total),1/1
# DeepLearning base models (used / total),1/1
# GLM base models (used / total),1/1
# DRF base models (used / total),2/2
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


In [22]:
print("\n--- LEADERBOARD DEI MODELLI (IL THRESHOLD DA BATTERE) ---")
lb = aml.leaderboard

print(lb.head(rows=lb.nrows))


--- LEADERBOARD DEI MODELLI (IL THRESHOLD DA BATTERE) ---
model_id                                                   mean_per_class_error    logloss      rmse       mse
StackedEnsemble_BestOfFamily_1_AutoML_3_20260618_165027                0.362376   0.74783   0.517861  0.26818
StackedEnsemble_AllModels_1_AutoML_3_20260618_165027                   0.362823   0.746092  0.516946  0.267233
GBM_5_AutoML_3_20260618_165027                                         0.371816   0.795604  0.528761  0.279589
GBM_grid_1_AutoML_3_20260618_165027_model_4                            0.374675   0.804527  0.535058  0.286287
GBM_2_AutoML_3_20260618_165027                                         0.378216   0.784021  0.526126  0.276809
GBM_4_AutoML_3_20260618_165027                                         0.380441   0.801199  0.531639  0.28264
DeepLearning_grid_3_AutoML_3_20260618_165027_model_2                   0.380532   0.775545  0.517028  0.267318
GBM_1_AutoML_3_20260618_165027                         

In [23]:
# metriche del modello vincitore sul Test Set
best_model = aml.leader
print("\nIl modello migliore (Leader) trovato da H2O AutoML è un:", best_model.algo)

print("\n--- PERFORMANCE DEL MODELLO VINCITORE SUL TEST SET ---")
performance = best_model.model_performance(test)
print(performance)


Il modello migliore (Leader) trovato da H2O AutoML è un: stackedensemble

--- PERFORMANCE DEL MODELLO VINCITORE SUL TEST SET ---
ModelMetricsMultinomialGLM: stackedensemble
** Reported on test data. **

MSE: 0.2849390573003636
RMSE: 0.533796831482132
LogLoss: 0.8390393529880856
Null degrees of freedom: 234
Residual degrees of freedom: 218
Null deviance: 507.3270311783366
Residual deviance: 394.3484959044002
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
1    2    3    Error     Rate
---  ---  ---  --------  --------
84   3    15   0.176471  18 / 102
11   44   15   0.371429  26 / 70
25   13   25   0.603175  38 / 63
120  60   55   0

### AutoML Results
- Errori Vittoria in casa: 18 errori / 102 guesses
- Errori Vittoria fuori casa: 26 errori / 70 guesses
- Errori Pareggio: 38 errori / 63 guesses
- Errori Totali: 82 errori / 235 guesses --> circa 34,8%
- Accuracy del modello Stacked Ensemble --> circa 65,2%

## Logistic Regression
Is the classic model used with sport datasets because it can also give the probability of win, draw and loss as a percentageù

- Need to standardize and scale the data in order to make it work properly

## Random Forest Classifier
Ensemble method based on decision trees, it is very good for small dataset just like this one

## Gradient Boosting
They are considered the state of the art of tree based algorythms